# CS 1066 Classnotes (20261021)

### As You Get Settled

1.  Start your CS1066 codespace. Go to `https://github.com/<your_username>/cs1066`, and either launch a new or restart the existing codespace for this repo.

2.  In this codespace, run `python3 grab1066.py m12a`. If you get an error, run `cd /workspaces/cs1066` in the terminal and then rerun the grab command.

3.  To get ready for the quiz, set your cellphone to silent, put your electronics away, and grab a pencil/pen. We'll post the Qwickly Attendance QR code after the quiz.

----
----

### Today's topics:

1.  Quiz 6
2.  A New Problem
3.  Knowledge Injection and RAG
4.  Building a Knowledge Base (Reading)
5.  Information Retrieval
6.  PSet #6 (shard m12b)

----
----

### Today's New Problem

Let's move from using an LLM to generate an application that uses a database to a situation where the combination of a database and an LLM is helpful.

On October 12, 2026, I asked GPT-5.6 Luna the following question:

----

**Prompt**

Databases and LLMs both answer user queries. What's the difference?

----

**LLM Response**

Both can answer questions, but they do so fundamentally differently:

| Aspect | Database | LLM |
|---|---|---|
| **What it stores** | Structured records, such as tables and rows | Learned patterns from large amounts of text/data |
| **How it answers** | Executes a precise query, such as SQL | Generates a likely response based on learned patterns and context |
| **Accuracy** | Usually exact for data it contains | Can be useful but may produce incorrect or fabricated information |
| **Data freshness** | Reflects its current stored data | Knowledge may be outdated unless connected to live sources |
| **Question types** | Best for well-defined facts, filtering, sorting, and aggregation | Best for language, explanation, summarization, interpretation, and open-ended questions |
| **Output** | Deterministic or mostly repeatable | Probabilistic; the same question may produce different wording or answers |
| **Reasoning** | Performs formally defined operations | Simulates reasoning through language generation, but does not guarantee logical correctness |

For example:

- **Database:** “How many students enrolled in CS1066 in 2025?”  
  It can count matching records exactly.
- **LLM:** “Explain why enrollment increased.”  
  It can propose an explanation, but it needs evidence and may be wrong.

In practice, they are often combined: an LLM interprets a natural-language question, converts it into a database query, retrieves the exact data, and then explains the result. (_This is our next topic!_)

----

### Live Demo

When is such a thing helpful?

**Attempt #1: Just Use a Chatbot**

-   Prompt: How many programming problem sets are in the course CS1066 at Harvard?

**Attempt #2: Augment the User's Prompt With Helpful Context**

-   User Prompt: How many programming problem sets are in the course CS1066 at Harvard?
-   Additional Context: You can find the course schedule at this URL: \<insert URL to tentative course schedule here>

> Why did the additional context help?

.

..

...

.... PLEASE DON'T SCROLL DOWN UNTIL TOLD TO DO SO

...

..

.

----

### Problem Summary

Goal: Improve the quality and accuracy of LLM responses.

The idea:

-   Provide an LLM with additional context that improves its ability to respond in a high-quality and accurate manner.
-   Provide this additional context with the user's prompt.

----
----

### Alternatives and Challenges

Retraining an LLM

-   Whether _continued pretraining_ (i.e., take an existing model and continue its pretraining on new text) or _full training_ (i.e., start with random parameters), this sort of training is expensive in terms of time and computing power.

Fine Tuning

-   This also starts with a pretrained model, but the goal is to adapt a model with existing language abilities to a relatively small, targeted dataset.
-   Approach often used when the LLM will be used for a specialized task or in a particular domain, e.g., for answering medical or legal questions.

> While retraining and fine tuning an LLM address the lack of specific knowledge, neither helps with the problem of _knowledge cutoff_.
>
> NOTE: This course won't teach you to train an AI model or fine tune one.

The simplest solution is _knowledge injection_ into the prompt.

----

### Knowledge Injection

**New questions:**

-   What knowledge should be injected into the user's prompt?
-   And how?

Remember the constraints on the LLM's context window:

-   You cannot add _all_ the latest knowledge into the prompt.
-   The size of the context window is large but limited.
-   Larger contexts increase the computational costs.

**New goal:**

Injecting the _right_ information into the prompt:

-   Keeps the LLM's computational costs low.
-   Focuses the LLM's response and reduces hallucinations.

This is where databases come back into the picture. With a well-structured query, you can identify and inject the "right" (i.e., relevant) information.

The result is called **an augmented prompt.**

----
----

### Retrieval Augmented Generation (RAG)

Working backward in the process flow:

-   The LLM _generates_ a response from an _augmented_ prompt. We're basically using the LLM for its language abilities and general knowledge.
-   The augmented prompt includes _relevant_ information from a knowledge base (i.e., a database). The information should help the LLM provide the user with a helpful answer.
-   A _retriever_ captures the user's prompt, uses it to create a query into the knowledge base, and selects the most-relevant retrieved information.

Same flow viewed in the forward direction in pseudocode:

In [ ]:
prompt = input("I'm a chatbot. How can I help you? ")

retrieved_info = retriever(prompt, knowledge_base)

augmented_prompt = f"{prompt}\n\nHere is some relevant information:\n{retrieved_info}"

response = llm(augmented_prompt)

Today, we'll focus on what it takes to build an effective retriever.

----

### Example Applications of RAG

RAG is working in many of the AI-infused apps that you use today:

-   **Your IDE.** It uses your files as its knowledge base (KB).
-   **Modern search engines.** Using the internet as the KB, a traditional search engine generates a list of search results, and the most relevant of these augment your prompt. An LLM answers this augmented prompt.
-   Many **corporate chatbots** with access to proprietary information.
-   **AI medical assistants**, which help doctors to find the most appropriate and up-to-date information in medical journals.
-   **Personal agents**, which use your personal data as the KB.

----
----

### Information Retrieval

Information retrieval is an important component of a RAG system with the following goals:

-   Return the most relevant documents/information
-   Withhold irrelevant ones

> NOTE: Going forward and without loss of generality, we'll think about the knowledge base (KB) as holding documents.

Architecture:

-   Input: User prompt
-   Components: Retriever and KB
-   Output: List of relevant documents (docs)

Critical steps in retriever process:

1.  Determine meaning of the prompt
2.  Identify docs in the KB relevant to the determined meaning
3.  Rank identified docs by relevance
4.  Return top-k docs

Techniques employed in these steps reflect best practices in search engine design and database retrieval.

> Today, we'll focus on steps 2-4 and return to step 1 next time.

----
----

### A Knowledge Base

To play with different retriever designs, we need a knowledge base. And preferably, one that contains information unknown to today's LLMs.

> Solution: I skipped forward in time and grabbed [the Harvard Women's Basketball game recaps](https://gocrimson.com/sports/womens-basketball/archives?search=&sport=wbball&season=0) for the seasons 2026-27 through 2035-36.
>
> See `m12/recaps`:
> 
> -   There are folders for each season.
> -   In each season folder, there's a Markdown file for each game played.
> -   Each file is a game recap, very much like the ones that Harvard Athletics produces.
> -   In each season folder, the `README.md` file contains the season's roster, schedule, and game results.
>
> The player names and personal details are fabricated. Only Coach Moore is an actual person, and her quotes in the game recaps are fabricated.

Producing this document set was an interesting exercise in using AI and it fits nicely into our problem-solving progression. So an aside before we return to information retrieval.

----

### Creating Season and Game Summaries

Our progression through problems in CS1066:

-   We started with problems where we mostly knew what we wanted (e.g., plotting EDGAR data and an AI player). The AI made implementation choices and a few, mostly inconsequential design decisions (e.g., how the webpage looked). These decisions were easy for us to see and change.
-   We moved to more complicated problems where we couldn't precisely specify what we wanted (i.e., ranking the employees of a startup). We wanted the AI to make design decisions so that we could explore the design space and refine our thinking. The challenge was in understanding what were the consequential design decisions that the AI made.
-   We are now adding a new twist. We know exactly what we want (i.e., made-up basketball seasons). But we don't want just the raw, numerical stats. We want narratives around each game and each season. And these narratives shouldn't sound mechanical, which is hard to precisely define.

In summary,

-   Our current task is numerically challenging, as there are lots of details to make match, but computers are good at such tasks **if programmed correctly**, which is what the AI will do for us.
-   In starting this work, I was more worried about the narratives. Would they be more than an English description of some dry statistics?

----

### My Prompt

See `m12/future.md` and notice the following:

-   I pointed the AI at a site containing many examples of the kinds of game summaries that I wanted it to produce. 
-   I then set the challenge (i.e., create game recaps for the next 10 seasons).
-   I followed this challenge with details of what I wanted to see for each season and each game. These details were far from complete, but I hoped that they'd be enough when coupled with the real examples.
-   I included a list of constraints on the fabricated data, which again were far from complete. Here, I was hoping that the AI would apply its general knowledge about basketball games.
-   Since the real game summaries came with lots of other stuff that isn't important to our RAG problem, I told the AI to avoid creating a number of things, like social media posts.
-   Since the dataset involves people, I directed the AI to avoid the names of any real people except Harvard's Women's Basketball coach.

The work involved in this prompt is large. The AI produced **305 recaps**. To avoid lots of wasted work, I concluded my prompt with the statement:

> Please make a plan for this work and check it with me before you start creating the Markdown files.

Unlike the employee-ranking problem, in which **iterating fast** was our goal, here I want the AI to produce a lot of material that:

-   I won't check closely when it's complete.
-   I won't monitor closely while it's being generated.
-   In fact, I'll do other things once I set the AI working. (The AI worked on this problem for over an hour.)

> So, I want to be sure that its plan is a good one.

----

### AI's Response

The AI's next action built on this staged idea:

-   It asked me several questions before it began any generation, which expanded on what I said about what it should and shouldn't include in the game summaries (e.g., statistics to track).
-   One of the most-important questions was, **would you like to generate a sample season to review?** This is a decomposition of the problem that I could have specified.

The procedure that the AI followed also recognized that I probably wouldn't check even this sample season. It gave me a short recap of the season (helping me decide if the season sounded reasonable) and pointed me to the Markdown files that might be helpful to review.

-   `README.md` is the numerical data on which all game summaries build. An error here would be catastrophic.
-   Two of the more exciting games, which should be engaging to the reader.
-   The game recaps that adhere closely to the script generator, which I should check to make sure that they are NOT be too boring to the reader.

> The AI worked to make my checking process easy!

----

### The AI's Plan

It wrote several programs:

-  one that simulated a season and generated coherent numerical data.
-  one that created formulaic game summaries from the numerical data.
-  one that validated the generated documents and confirmed that they adhered to the prompts and (the AI-generated) constraints.

It also "hand-polished ... nine big games," which means that the LLM rewrote the formulaic articles.

----
----

### Information Retrieval: Keyword Search

Intuitively, you all know how an internet search site worked **before LLMs**:

> You typed a list of keywords, and the search engine returned the webpages containing those keywords.

Inside the engine,

-   There was **an inverted index** that mapped terms to webpages, which made search fast. This index was regularly updated by crawling the entire (reachable) web.
-   Creating this index required the conversion of every webpage into **a bag of words**, which is a text featurization technique in NLP that converts text into numeric vectors based on a page's word counts.
-   Instead of highly ranking those webpages with more occurrences of your search terms, people invented **sophisticated ranking algorithms** like:
    - TF-IDF with cosine similarity
    - BM25

> NOTE: We won't test you on these algorithms, but please feel free to investigate how they work.

What we care about is that **keyword search** algorithms excel when you want **the documents that contain a specific term** (e.g., a product name or serial number).

----

### Setup (Keyword Search)

I had AI generate the Python program `keyword_search.py`. I made a few small interface improvements to the code.

Usage:

-   Create a virtual environment and install dependencies in `requirements.txt`.
-   Run `python3 keyword_search.py`
-   By default, it uses the KB in the folder `recaps`. You can change this default by specifying the path to your own KB as the first command-line parameter (see `python3 keyword_search.py --help`).
-   When running, the program asks you for a query and then a ranking method. It defaults to BM25, which is generally superior to TF-IDF.
-   Type `q` when you want to exit the program.

----

### Demo (Keyword Search)

This approach works well for queries like:

-   "Petrakis double double"

It doesn't work so well for queries like:

-   "Who led Harvard in scoring?"
-   "Harvard top scorer"
-   "Most points for Harvard"

These queries are semantically related, but they find different subsets of documents. This opens the door for a different approach.

----
----

### Information Retrieval: Semantic Search

Rather than performing a string match, semantic search attempts to **match on meaning**.

-   e.g., "Who led Harvard in scoring?" = "Harvard top scorer"

Approach, in general:

1.  A machine learning model converts the query and each KB document into **vector embeddings**.
2.  Vector embeddings are **long lists of numbers** that represent **points in a high-dimensional space**.
3.  If two points are **near each other**, then their **meanings are similar**.
4.  **Ranking documents** is a process of sorting the distance from the query to its nearest neighboring documents.

> NOTES:
>
> -   You can't compare points in different spaces.
> -   There are numerous methods that find similar vectors (e.g., cosine similarity and k-nearest neighbors).
> -   We won't test you on the different methods. You simply need to understand the general approach.
> -   We'll let AI use off-the-shelf embedding models and distance measurements.

----

### Setup (Semantic Search)

I had AI generate the Python program `semantic_search.py`. I made a few small interface improvements to the code.

Some details of the implementation:

1.  The program **chunks** the recap documents into sentences. Why?
    - Meaning is often more accurate at a chunk granularity.
    - Better performance from small embedding models.
2.  It embeds the input query and each chunk.
3.  It uses cosine similarity to find the most similar chunks.
4.  It groups matching chunks by source document.
5.  It returns a ranked list of recap documents, where the score of a document is its best chunk score.

Usage:

-   Create a virtual environment and install dependencies in `requirements.txt`.
-   Run `python3 semantic_search.py`
-   By default, it uses the KB in the folder `recaps`. You can change this default by specifying the path to your own KB as the first command-line parameter (see `python3 semantic_search.py --help`).
-   When running, the program first loads an embedding model from Hugging Face (anonymously). It then asks you for a query.
-   Type `q` when you want to exit the program.

----

### Demo (Semantic Search)

How well does this approach work for the following queries? Compare with `keyword_search.py`.

-   "70-59"
-   "Who led Harvard in scoring?"
-   "Which game included a late fourth-quarter comeback?"
-   "Harvard won despite poor ball security"
-   "A close game decided in the final minute."
-   Write a query for a game on a particular date.

----

### Limitations of the Semantic Search implementation

- Embeddings are kept only in memory (and recomputed every time the program starts).
- The chunking is fairly simplistic.
- You can do better.

----
----

### Information Retrieval: Hybrid Search

Given the advantages and disadvantages of keyword and semantic search, most retrievers combine these complementary approaches. The result is called **hybrid search**.

A topic for next time...

----
----

### PSet #6 Will Be Assigned on Monday

But looking ahead, it focuses on understanding the search functions that we just covered. 

The pset is not hard, but asks you to think.

-   Encourages you to use AI, but be efficient in your use of it.
-   No "brute-force" mentality!

----
----

### Commit Your Notes and Edits

Don't forget to review, commit, and push your edits so that you've updated your GitHub repo with today's work.

*   `git status` to review what's new and what's been edited.
*   `git add <list_of_files>` to indicate what you want committed.
*   `git commit -m "<your_commit_message>"` to commit the changes to the codespace's filesystem.
*   `git push` to push those committed changes to your GitHub repo.
*   `git status` to verify that you haven't forgotten anything.

Version 20261020